# t-SNE e UMAP

**Objetivo:** comparar a projeção **linear** da PCA com a **não linear** do t-SNE no conjunto de dígitos manuscritos, ver os dígitos se separarem em ilhas, e experimentar o efeito da perplexidade. UMAP entra como opcional.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

In [ ]:
from sklearn.datasets import load_digits

digitos = load_digits()
# subamostra para o t-SNE rodar rapido
rng = np.random.RandomState(SEMENTE)
sel = rng.choice(len(digitos.data), 600, replace=False)
X = digitos.data[sel]      # 64 dimensoes (imagens 8x8)
y = digitos.target[sel]
print("X:", X.shape, "| digitos de 0 a 9")

## 1. PCA linear × t-SNE não linear

A PCA projeta no plano de maior variância; o t-SNE preserva a **vizinhança local**. Lado a lado, os dígitos que a PCA mistura o t-SNE separa em ilhas.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

coords_pca = PCA(n_components=2).fit_transform(X)
coords_tsne = TSNE(n_components=2, perplexity=30, init="pca",
                   random_state=SEMENTE).fit_transform(X)

from plotly.subplots import make_subplots
figura = make_subplots(rows=1, cols=2, subplot_titles=("PCA (linear)", "t-SNE (nao linear)"))
figura.add_trace(go.Scatter(x=coords_pca[:, 0], y=coords_pca[:, 1], mode="markers",
                            marker=dict(color=y, colorscale="Rainbow", size=5, showscale=False),
                            text=y), row=1, col=1)
figura.add_trace(go.Scatter(x=coords_tsne[:, 0], y=coords_tsne[:, 1], mode="markers",
                            marker=dict(color=y, colorscale="Rainbow", size=5, showscale=False),
                            text=y), row=1, col=2)
figura.update_layout(height=420, showlegend=False, margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 2. O efeito da perplexidade

A perplexidade regula quantos vizinhos cada ponto considera. Valores muito baixos fragmentam; muito altos borram. Comparamos três.

In [ ]:
figura = make_subplots(rows=1, cols=3, subplot_titles=("perplexidade 5", "30", "50"))
coluna = 1
for perp in [5, 30, 50]:
    coords = TSNE(n_components=2, perplexity=perp, init="pca",
                  random_state=SEMENTE).fit_transform(X)
    figura.add_trace(go.Scatter(x=coords[:, 0], y=coords[:, 1], mode="markers",
                                marker=dict(color=y, colorscale="Rainbow", size=4)),
                     row=1, col=coluna)
    coluna += 1
figura.update_layout(height=340, showlegend=False, margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 3. (Opcional) UMAP

Se a biblioteca `umap-learn` estiver instalada (no Colab, um `!pip install umap-learn` resolve), a célula abaixo roda o UMAP; senão, avisa e segue.

In [ ]:
try:
    import umap
    reducao = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEMENTE)
    coords_umap = reducao.fit_transform(X)
    figura = go.Figure(go.Scatter(x=coords_umap[:, 0], y=coords_umap[:, 1], mode="markers",
                                  marker=dict(color=y, colorscale="Rainbow", size=5)))
    figura.update_layout(title="UMAP dos digitos", height=420, showlegend=False,
                         margin=dict(l=10, r=10, t=50, b=10))
    figura.show()
except Exception as erro:
    print("umap-learn nao disponivel — pulando.")
    print("para instalar no Colab: !pip install umap-learn")
    print("detalhe:", type(erro).__name__)

## Exercício

No mapa t-SNE, dois grupos de dígitos aparecem bem afastados. Você pode concluir que esses dígitos são "mais diferentes" entre si do que dois grupos próximos? Por quê?

<details><summary>Ver resposta</summary>

**Não.** O t-SNE preserva a estrutura **local** (quem é vizinho de quem), não as distâncias **globais**. A separação entre dois grupos no mapa é amplamente arbitrária — o algoritmo é livre para posicionar ilhas distantes sem que isso reflita a distância real no espaço de 64 dimensões. Para comparar o quão diferentes são dois grupos, é preciso medir no espaço original, não no mapa.

</details>